# 03_compare — 품질·계산량·효율 비교와 판정 (RF-06)

[RF-06 계획서](../../../docs/ml/VERSION_POLICY_EFFICIENCY_EXPERIMENT.md) 5·6절. 학습하지 않고 고르지 않는다. 봉인된 plan과 끝난 round만 읽는다.

- **품질(주):** round마다 dev source-macro-F1, 마지막 round의 test source-macro-F1(P0 = M0).
- **판정:** 질문 그룹 paired bootstrap 95% 구간. P1 − P2의 하한이 −.01보다 크면 "비열등", P1·P2 − P0가 +.01 초과이고 하한 > 0이면 "효과 있음".
- **계산량:** 누적 optimizer step, T4 학습 시간, 처리 토큰. P2/P1 배수와 "P2 마지막 − .01"에 처음 닿은 round.
- **출력:** Drive `compare/`(results.json, tables, figures, REPORT-draft.md, results-for-review.zip). 질문 원문은 넣지 않는다.

**런타임: CPU로 충분하다.** 몇 분. 중간에 상황을 보려면 `PARTIAL = True`(끝난 round만 집계, `compare-partial/`에 씀).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/version-policy/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'version-policy-source.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/version-policy-source')
shutil.rmtree(REPO, ignore_errors=True)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'

In [ ]:
CODE = REPO / 'ml/notebooks/version_policy'
DATA = RUN / 'data'
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'), PYTHONUNBUFFERED='1', TRANSFORMERS_VERBOSITY='error')

def run_logged(script, args):
    """Run a script in a fresh process (it sees the packages installed above) and stream its log here."""
    proc = subprocess.Popen([sys.executable, str(CODE / script), *map(str, args)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=ENV, cwd=str(CODE))
    for line in proc.stdout:
        if 'Warning' not in line and 'warn(' not in line:
            print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

assert (DATA / 'manifest.json').exists(), 'Drive에 data/(로컬 build 결과)를 먼저 올리세요. README 참고'
print(json.loads((DATA / 'manifest.json').read_text())['rows'])

In [ ]:
PARTIAL = False
OUT = RUN / ('compare-partial' if PARTIAL else 'compare')
run_logged('compare.py', ['--run-dir', RUN, '--data-dir', DATA, '--output-dir', OUT] + (['--allow-partial'] if PARTIAL else []))
print((OUT / 'REPORT-draft.md').read_text())

In [ ]:
from IPython.display import Image, display
for png in sorted((OUT / 'figures').glob('*.png')):
    display(Image(filename=str(png)))